# `load_processed_documents_from_gcs` — রান্না-করা data তোলা

এই function-টা আগেরটার প্রায় যমজ। কাঠামো এক, কাজের চরিত্র আলাদা। আগেরটা কাঁচা text পড়ত। এটা আগেই রান্না-করা data তুলে আনে।

পার্থক্যগুলোতেই মূল শিক্ষা, তাই সেদিকেই বেশি সময় দেব। আগের notebook-এর মতো, cloud ছাড়া যা বোঝা যায় সেগুলো নিচের cell-এ চলবে। একদম শেষেরটাই আসল GCS call।

---

## প্রথমে বড় ছবিটা — দুই zone-এর গল্প

তোমার bucket-এ আসলে দুইটা এলাকা। একটায় মানুষ যা upload করে, আরেকটায় `processor.py` যা লিখে রাখে।

```
myhr-policies/
├── hr_policies/raw/          ← .txt, .pdf, .docx  (মানুষ যা upload করে)
├── hr_policies/processed/    ← .json              (processor.py যা লেখে)
├── noise_policies/raw/
└── noise_policies/processed/
```

`load_documents_from_gcs()` পড়ে **raw zone**। এই function পড়ে **processed zone**। মাঝখানে দাঁড়িয়ে আছে `processor.py` (05)। সে সব format থেকে text বের করে একরকম JSON বানিয়ে রাখে।

এই ভাগটাকে বলে **medallion architecture**, বা raw/processed zone pattern। Data engineering-এ খুব প্রচলিত। যুক্তিটা সহজ: ভারী আর ভঙ্গুর কাজটা — PDF parse, OCR, encoding ঝামেলা — একবারই করো, ফল জমিয়ে রাখো, তারপর যতবার খুশি সস্তায় পড়ো। Chunk size বদলে ১০ বার ingestion চালালে ১০ বার PDF parse করতে হয় না।

Docstring-এ ঠিক এই কথাটাই লেখা: "No parsing here"। শেষ বাক্যটা সবচেয়ে জরুরি — **এটাই তোমার আসল pipeline (09) যে function-টা ডাকে।** আগের function মূলত raw zone দেখা আর debug-এর জন্য রয়ে গেছে।

এই নোটের উদাহরণে path লেখা `hr_policies/processed/`। তোমার `config.py`-তে আসল prefix দুইটা: `processed/myhr-rag-data/` আর `processed/other-data/`। শেষের cell ওই দুইটাই ব্যবহার করে।

---

## অংশ ১ — Signature-এ নতুন যা

```python
prefixes: tuple[str, ...] = (config.PROCESSED_HR_PREFIX, config.PROCESSED_NOISE_PREFIX),
```

আগে ছিল একটা `prefix`, একটা string। এখন `prefixes`, একটা tuple। কারণ দুই জায়গা থেকে পড়তে হবে: HR policy আর noise document।

### `tuple[str, ...]` এর `...` মানে কী

ওই তিনটা ডট Python-এর `Ellipsis`। Tuple-এর type hint-এ এর একটা বিশেষ অর্থ আছে:

```python
tuple[str, str]      # ঠিক দুইটা string, এর কম-বেশি নয়
tuple[str, ...]      # যেকোনো সংখ্যক string
```

দ্বিতীয়টা বেছে নেওয়ার কারণ ভবিষ্যতের জন্য। তৃতীয় একটা zone যোগ হলে — ধরো `PROCESSED_LEGAL_PREFIX` — signature বদলাতে হবে না।

নিচের cell শুধু ডাকের রূপ দেখায়। আসল function শেষের cell-এ import হবে।

In [ ]:
# শুধু tuple কীভাবে পাঠানো যায়, সেটা দেখানো। আসল function শেষের cell-এ import হয়।
def load_processed_documents_from_gcs(
    bucket_name: str = "hr-policies-bucket",
    prefixes: tuple[str, ...] = ("hr_policies/processed/", "noise_policies/processed/"),
):
    return bucket_name, prefixes

print(load_processed_documents_from_gcs())
print(load_processed_documents_from_gcs(prefixes=("hr_policies/processed/",)))
print(load_processed_documents_from_gcs(prefixes=("hr/", "noise/", "legal/")))

### List নয়, tuple কেন

এটা শখ না, একটা নিরাপত্তা। **Default argument কখনো mutable হওয়া উচিত নয়।** এটা Python-এর সবচেয়ে বিখ্যাত ফাঁদ।

Default value function define হওয়ার সময় **একবারই** তৈরি হয়, প্রতি call-এ নয়। List mutable, তাই সেই একই list call-এর পর call-এ জমতে থাকে। Tuple immutable, তাই ভেতরে `.append()` করতে গেলে সাথে সাথে `AttributeError`।

নিচের cell দুটোই দেখায়। প্রথম function-এর দ্বিতীয় ডাকে আগের `"x"` এখনও আছে।

In [ ]:
def bad(items=[]):
    items.append("x")
    return items

print(bad())
print(bad())

def good(items=()):
    try:
        items.append("x")
    except AttributeError as error:
        print(type(error).__name__ + ":", error)

good()

### Tuple লেখার সময় একটা ফাঁদ

Notebook-এ একটামাত্র prefix দিতে চাইলে শেষে একটা comma লাগে।

```python
load_processed_documents_from_gcs(prefixes=("hr_policies/processed/",))   # comma আছে, এটা tuple
load_processed_documents_from_gcs(prefixes=("hr_policies/processed/"))    # comma নেই, এটা string
```

দ্বিতীয়টায় বন্ধনী শুধু grouping করছে। ভেতরে পড়ে আছে একটা সাধারণ string। তারপর `for prefix in prefixes` সেই string-এর **প্রতিটা অক্ষরের** উপর ঘোরে — `'h'`, `'r'`, `'_'` — আর প্রতিবার `list_blobs(prefix='h')` কল করে। Error দেয় না। শুধু আজব ফল দেয়। একটা element-এর tuple-এ শেষের comma বাধ্যতামূলক।

In [ ]:
with_comma = ("hr_policies/processed/",)
without_comma = ("hr_policies/processed/")

print("comma আছে   ", type(with_comma).__name__, with_comma)
print("comma নেই   ", type(without_comma).__name__)
print("অক্ষর ধরে ঘুরল:", list(without_comma))

---

## অংশ ২ — Client, হুবহু আগের মতো

```python
client = storage.Client(project=config.PROJECT_ID)
bucket = client.bucket(bucket_name)
```

আগের ব্যাখ্যাই খাটে। `client.bucket()` কোনো network call নয়, শুধু local reference। নাম ভুল হলে এই line চুপ করে থাকে। Error আসে পরে, `list_blobs()` যখন server-এ যায়।

একটা জিনিস লক্ষ্য করো: **দুইটা prefix-এর জন্য একটাই client, একটাই bucket।** Bucket একই, শুধু ভেতরের path আলাদা। প্রতিটা prefix-এর জন্য নতুন client বানানো অপচয় — client তৈরির সময় credential resolve করা তুলনামূলক ভারী কাজ।

---

## অংশ ৩ — Nested loop

```python
documents = []
for prefix in prefixes:
    for blob in bucket.list_blobs(prefix=prefix):
```

এটাই নতুন কাঠামো: **loop-এর ভেতরে loop**।

- বাইরের loop চলে prefix-এর সংখ্যাবার। Default-এ ২ বার।
- ভেতরের loop প্রতিবার চলে ওই prefix-এ যত blob, ততবার।

### `bucket.list_blobs(prefix=prefix)`

এই লাইন বাইরের loop-এর প্রতি চক্করে একবার চলে। প্রথমবার `prefix` হয় HR zone-এর path, দ্বিতীয়বার noise zone-এর path। তাই দুইবার দুইটা আলাদা listing। একই bucket, আলাদা প্রশ্ন।

`prefix=prefix` দেখতে অদ্ভুত, দুই পাশে একই শব্দ। দুই পাশের কাজ আলাদা।

- বাঁ পাশের `prefix` হলো library-র parameter-এর নাম। "নাম কোন অক্ষর দিয়ে শুরু, সেই blob গুলো দাও।"
- ডান পাশের `prefix` হলো এই চক্করের string, বাইরের loop থেকে আসা।

Keyword বাধ্যতামূলক। `list_blobs`-এর প্রথম positional argument `max_results`, তারপর `page_token`, তারপর `prefix`। তাই `bucket.list_blobs(prefix)` লিখলে সেই string নাম-ফিল্টার হয় না — সে চলে যায় `max_results`-এ, আর listing ভেঙে যায়। নাম মিলে যাওয়াটা ভুল না। বাঁ পাশ parameter, ডান পাশ মান।

কী ফেরত আসে: Python list না, একটা **iterator**। প্রতিটা item একটা `Blob`। এই মুহূর্তে blob-এর কাছে আছে নাম আর হালকা metadata — size, content type, কখন বদলেছে। JSON-এর ভেতরের লেখা এখনও নামেনি। লেখা আসে পরের লাইনে, `download_as_text()`। Listing সস্তা। Download দামি। ৫০টা file হলে list হয় কয়েকটা page-এ, download হয় ৫০ বার।

Network কখন হয়: `list_blobs(...)` লিখলেই request যায় না। `for` যখন প্রথম blob চায়, তখন প্রথম page আসে। `page_size` না দিলে server এক পাতায় সাধারণত ১০০০টা পর্যন্ত দেয়। তার বেশি হলে iterator নিজে পরের page টানে। তুমি loop-এ শুধু একটা একটা blob পাও, পুরো page নিজে জোড়া লাগাতে হয় না।

`prefix` মানে folder খোলা না। **যে নাম এই অক্ষর দিয়ে শুরু।** `delimiter` এখানে দেওয়া হয়নি, তাই ভেতরের subfolder-ও আসে — `processed/myhr-rag-data/2024/old.json`-ও মিলবে। `delimiter="/"` দিলে শুধু ওই স্তরের সরাসরি নাম আসত, ভেতরের folder আলাদা prefix হয়ে। এখানে সেটা চাই না। Processed zone-এর সব JSON চাই।

শেষে `/` না থাকলে পাশের নামও ঢুকে পড়ে। `processed/myhr` মিলত `processed/myhr-rag-data/` আর `processed/myhr-old/` দুইটাতেই।

প্রতিবার `list_blobs()` **নতুন iterator** দেয়। ডাকটা বাইরের loop-এর ভেতরে আছে বলে দ্বিতীয় prefix আলাদা request। একটা iterator একবার শেষ হলে খালি, দ্বিতীয়বার ঘোরানো যায় না। এখানে সেই সমস্যা নেই, কারণ prefix বদলালে আবার নতুন করে ডাকা হচ্ছে।

খালি string দিলে পুরো bucket উঠে আসে। তোমার default দুইটা নির্দিষ্ট path, তাই পুরো bucket আসে না।

Console থেকে folder বানালে মাঝে মাঝে একটা ফাঁকা blob আসে, নাম শেষ হয় `/` দিয়ে। সেও এই listing-এ ঢোকে। পরের লাইনের `.endswith(".json")` তাকে সরিয়ে দেয়।

নিচের cell আসল cloud না। একই নিয়ম ছোট ডেটায়: keyword না দিলে মান ভুল জায়গায় যায়, `/` সহ prefix শুধু ওই নামগুলো রাখে, আর page করে একটা একটা blob বেরোয়।

In [ ]:
def list_blobs(max_results=None, page_token=None, prefix=None, page_size=2):
    """library-র মতো: prefix keyword। page_size মানে এক পাতায় কয়টা।"""
    if not isinstance(prefix, str):
        print(f"  max_results={max_results!r}  prefix={prefix!r}  → নাম ফিল্টার হয়নি")
        return []
    matched = [name for name in catalog if name.startswith(prefix)]
    print(f"  prefix={prefix!r}  মিলল {len(matched)}টা")
    for start in range(0, len(matched), page_size):
        page = matched[start:start + page_size]
        print(f"    page {start // page_size + 1}: {page}")
        yield from page


catalog = [
    "hr_policies/processed/wfh.json",
    "hr_policies/processed/leave.json",
    "hr_policies/processed/2024/old.json",
    "hr_policies/processed/",
    "hr_policies_backup/secret.json",
    "noise_policies/processed/canteen.json",
    "noise_policies/processed/parking.json",
]

print("keyword ছাড়া — string চলে যায় max_results-এ")
list(list_blobs("hr_policies/processed/"))  # iterate না করলে generator চলেই না

print()
for prefix in ("hr_policies/processed/", "noise_policies/processed/"):
    print("নতুন listing")
    for blob_name in list_blobs(prefix=prefix):
        print("   blob", blob_name)

`documents = []` আছে **দুইটা loop-এরই বাইরে**। দুই zone-এর ফল একই ঝুড়িতে জমছে। ভেতরে রাখলে দ্বিতীয় prefix শুরুর সময় প্রথমটার সব কাজ মুছে যেত। নিচের cell-এ দুটো রাখার জায়গা পাশাপাশি চালাও।

In [ ]:
zones = {
    "hr_policies/processed/": ["wfh.json", "leave.json", "travel.json"],
    "noise_policies/processed/": ["canteen.json", "parking.json"],
}

documents = []
for prefix, names in zones.items():
    for name in names:
        documents.append(name)
print("বাইরে রাখলে:", documents)

for prefix, names in zones.items():
    documents = []
    for name in names:
        documents.append(name)
print("ভেতরে রাখলে শেষে থাকে:", documents)

ফলে return-এ পাবে **একটা মিশ্র list** — HR আর noise document পাশাপাশি, আলাদা করা নেই।

### এটা ইচ্ছাকৃত, এখানেই project-এর মূল পরীক্ষা

Noise document ভুল করে ঢোকেনি। ওগুলো **বসানো হয়েছে** — canteen menu, parking rules, office picnic notice — যাতে vector store-এ সত্যিকারের বিভ্রান্তি থাকে। কেউ জিজ্ঞেস করলে "আমি সপ্তাহে কয়দিন বাসা থেকে কাজ করতে পারি?", retriever-কে WFH policy বেছে নিতে হবে, canteen menu নয়।

Noise ছাড়া accuracy মাপা কঠিন। সব chunk প্রাসঙ্গিক হলে ভুল করার সুযোগই থাকে না। এই একটা nested loop তোমার evaluation-এর ভিত্তি।

কোন category কতটা এলো, সেটা `Counter` দিয়ে এক নজরে দেখা যায়। নিচে নমুনা। আসল সংখ্যা শেষের cell-এ, cloud থেকে।

In [ ]:
from collections import Counter

categories = ["Work From Home", "Leave", "Leave", "Canteen", "Parking"]
print(Counter(categories))

---

## অংশ ৪ — ফিল্টার, শুধু extension বদলেছে

```python
if not blob.name.endswith(".json"):
    continue
```

আগে ছিল `.txt`, এখন `.json`। Processed zone-এ সবকিছুই JSON, মূল format যাই হোক।

এখানেই raw zone-এর সাথে বড় পার্থক্য। Raw zone-এ `.pdf` বা `.docx` থাকলে আগের function সেগুলো এড়িয়ে যেত — ও পড়তেই পারত না। Processed zone-এ ওই PDF এখন একটা JSON, কারণ `processor.py` text বের করে JSON-এ লিখে রেখেছে।

মানে এই function আসলে **বেশি document পায়**, যদিও ফিল্টার দেখতে একই রকম। Raw-তে ২০টা file আর processed-এ ২০টা JSON না মিললে ঘাবড়ানোর কিছু নেই। বরং হুবহু মিলে যাওয়াটাই মাঝে মাঝে সন্দেহের জায়গা — কোনো file parse হয়নি কি না, সেটা আলাদা করে দেখতে হয়।

In [ ]:
names = [
    "hr_policies/processed/work_from_home_policy.json",
    "hr_policies/processed/handbook.json",
    "hr_policies/raw/handbook.pdf",
    "hr_policies/processed/",
]

for name in names:
    if not name.endswith(".json"):
        print("continue", name)
        continue
    print("keep    ", name)

---

## অংশ ৫ — এই function-এর হৃদয়

```python
record = json.loads(blob.download_as_text())
```

এক লাইনে দুইটা কাজ, ভেতর থেকে বাইরে।

**ধাপ ১.** `blob.download_as_text()` network call। JSON file নামিয়ে আনে **একটা string** হিসেবে। এই মুহূর্তে Python-এর কাছে এটা শুধু অক্ষরের সারি:

```python
'{"source": "wfh.txt", "text": "WORK FROM HOME...", "policy_category": "Work From Home"}'
```

**ধাপ ২.** `json.loads(...)` সেই string-কে **Python dict**-এ বানায়:

```python
{
    "source": "wfh.txt",
    "text": "WORK FROM HOME...",
    "policy_category": "Work From Home",
    "raw_gcs_path": "gs://myhr-policies/hr_policies/raw/wfh.txt",
}
```

এখন `record["text"]` দিয়ে value তোলা যায়। এটা আর string নয়, সত্যিকারের dict।

In [ ]:
import json

raw = (
    '{"source": "wfh.txt", "text": "WORK FROM HOME...", '
    '"policy_category": "Work From Home", '
    '"raw_gcs_path": "gs://myhr-policies/hr_policies/raw/wfh.txt", '
    '"active": true, "note": null}'
)
print("নামানোর পর:", type(raw).__name__)

record = json.loads(raw)
print("loads-এর পর:", type(record).__name__)
print(record["text"])
print("true/null →", record["active"], record["note"])
print("number-এর মতো key →", list(json.loads('{"1": "a"}').keys()), type(list(json.loads('{"1": "a"}').keys())[0]).__name__)
print(json.dumps(record))

### `loads`-এর শেষের `s` মানে string

আগের নোটে বলেছিলাম। এখানেই ব্যবহার হচ্ছে, তাই আরেকবার:

| Function | কী নেয় | কখন |
|---|---|---|
| `json.loads(s)` | **string** | এখানে, GCS text নামানোর পর |
| `json.load(f)` | file object | `open()` দিয়ে খোলা local file |
| `json.dumps(obj)` | → string দেয় | `processor.py`, GCS-এ লেখার আগে |

মনে রাখার সহজ উপায়: `s` মানে string। File থাকলে `s` নেই।

### JSON আর Python dict এক জিনিস নয়

দেখতে প্রায় এক। JSON হলো **text format** — file-এ লেখা যায়, network-এ পাঠানো যায়। Dict হলো **memory-তে থাকা Python object**। `loads` সেতু।

পারাপারের সময় কিছু জিনিস বদলে যায়:

| JSON-এ | Python-এ |
|---|---|
| `true` / `false` | `True` / `False` |
| `null` | `None` |
| key সবসময় string | JSON থেকে এলে key string-ই থাকে |

শেষ row মনে রেখো। JSON-এ সংখ্যার মতো key লিখলেও ফেরত আসে string হয়ে। উপরের cell-এ `true`, `null`, আর `"1"` তিনটাই দেখা যাচ্ছে।

---

## অংশ ৬ — Document বানানো, উৎস আলাদা

```python
documents.append(Document(
    page_content=record["text"],
    metadata={
        "source": record["source"],
        "policy_category": record["policy_category"],
        "gcs_path": record["raw_gcs_path"],
    },
))
```

আউটপুটের আকার আগের function-এর সাথে **হুবহু এক** — একই তিনটা metadata key, একই `page_content`। এটা দুর্ঘটনা নয়। Downstream-এর splitter, vector store, retriever জানে না document কোন function থেকে এলো। দুইটা function **interchangeable**।

Value কোথা থেকে আসছে, সেখানেই পার্থক্য:

| Metadata | আগের function (raw) | এই function (processed) |
|---|---|---|
| `source` | `blob.name.rsplit(...)` — নিজে হিসাব করত | `record["source"]` — পড়া মাত্র |
| `policy_category` | `extract_policy_category(text)` — নিজে parse করত | `record["policy_category"]` — পড়া মাত্র |
| `gcs_path` | f-string দিয়ে নিজে বানাত | `record["raw_gcs_path"]` — পড়া মাত্র |

তিন জায়গাতেই হিসাব করা থেকে পড়ে নেওয়ায় বদল। তাই docstring-এ "No parsing here"। কোনো `rsplit` নেই, কোনো f-string নেই, `extract_policy_category`-র কোনো ডাক নেই। সব কাজ আগেই `processor.py` সেরে রেখেছে।

### `raw_gcs_path` নামটা খেয়াল করো

JSON-এর key `gcs_path` নয়, `raw_gcs_path`। Metadata-তে বসছে `gcs_path` নামে। ইচ্ছাকৃত। Path-টা দেখায় **মূল raw file কোথায়**, JSON-টা কোথায় নয়।

কোনো উত্তর সন্দেহজনক লাগলে তুমি আসল PDF বা `.txt` খুলতে চাইবে, processor-এর বানানো JSON নয়। এই একটা নাম পরে অনেক সময় বাঁচায়।

In [ ]:
class Document:
    def __init__(self, page_content, metadata):
        self.page_content = page_content
        self.metadata = metadata


record = {
    "source": "wfh.txt",
    "text": "WORK FROM HOME...",
    "policy_category": "Work From Home",
    "raw_gcs_path": "gs://myhr-policies/hr_policies/raw/wfh.txt",
}

documents = []
documents.append(Document(
    page_content=record["text"],
    metadata={
        "source": record["source"],
        "policy_category": record["policy_category"],
        "gcs_path": record["raw_gcs_path"],
    },
))
print(documents[0].page_content)
print(documents[0].metadata)

---

## পূর্ণ trace

ধরো দুই zone-এ যথাক্রমে ৩টা আর ২টা JSON। `documents` বাইরে থাকায় প্রথম zone-এর কাজ দ্বিতীয় চক্করে মুছে যায় না।

| পর্যায় | কী হচ্ছে | `documents` |
|---|---|---|
| বাইরের loop, ১ম চক্কর | `prefix = "hr_policies/processed/"` | `[]` |
| ভেতরের loop ×3 | ৩টা JSON নামল, parse হলো | দৈর্ঘ্য 3 |
| বাইরের loop, ২য় চক্কর | `prefix = "noise_policies/processed/"` | দৈর্ঘ্য 3, মুছল না |
| ভেতরের loop ×2 | ২টা JSON নামল | দৈর্ঘ্য 5 |
| `return documents` | — | **5টা Document** |

এরপর এই ৫টা যাবে `splitter.py` (06)-এ, chunk-এ ভাঙবে, তারপর embedding, তারপর vector store।

In [ ]:
zones = [
    ("hr_policies/processed/", ["wfh.json", "leave.json", "travel.json"]),
    ("noise_policies/processed/", ["canteen.json", "parking.json"]),
]

documents = []
for prefix, names in zones:
    print(f"বাইরের loop  {prefix:<32} শুরুতে {len(documents)}")
    for name in names:
        documents.append(name)
    print(f"              ভেতরের loop শেষে দৈর্ঘ্য {len(documents)}")

print("return", len(documents), "টা Document")

---

## দুইটা function পাশাপাশি

| | `load_documents_from_gcs` | `load_processed_documents_from_gcs` |
|---|---|---|
| পড়ে | raw zone | processed zone |
| Format | `.txt` মাত্র | `.json` (সব format-ই এখানে) |
| Prefix | একটা | tuple, একাধিক |
| Loop | একটা | nested |
| Metadata | নিজে হিসাব করে | JSON থেকে পড়ে |
| Noise আসে? | না | হ্যাঁ, ইচ্ছাকৃত |
| কে ডাকে | manual / debug | **ingestion pipeline (09)** |

---

## যেখানে ভাঙতে পারে

এই function-এ আগেরটার তুলনায় একটা নতুন ঝুঁকি ঢুকেছে: data-র উপর অন্ধ নির্ভরতা।

`record["text"]` মানে "আমি নিশ্চিত এই key আছে"। না থাকলে `KeyError`, আর পুরো ingestion ওখানেই থেমে যায়। `processor.py`-র কোনো পুরনো version যদি `content` নামে লিখে থাকে, বা কোনো file অর্ধেক JSON হয়ে থাকে — pipeline crash।

`json.loads()` ভাঙা JSON পেলে `JSONDecodeError` দেয়। সেটাও পুরো loop থামায়।

নিচের cell দুটো error-ই ধরে দেখায়, যাতে notebook থেমে না যায়। আসল file-এ এই `try` নেই — ওখানে error উঠলে function থেমেই যায়।

In [ ]:
import json

record = {"source": "wfh.txt"}
try:
    record["text"]
except KeyError as error:
    print("KeyError:", error)

try:
    json.loads("{broken")
except json.JSONDecodeError as error:
    print("JSONDecodeError:", error.msg)

এগিয়ে যাওয়ার আগে একবার দেখে নাও processed zone-এ কোন key আসছে। `text`, `source`, `policy_category`, `raw_gcs_path` — চারটাই থাকা দরকার। না মিললে সমস্যা এই function-এ নয়, `processor.py`-তে।

তারপর সেই একই অভ্যাস: কয়টা document এলো। `0` মানে processed zone খালি — `processor.py` এখনো চালানো হয়নি, অথবা prefix-এ টাইপো। Vector store খালি নিয়ে এগিয়ে "retriever কিছু পাচ্ছে না কেন" খোঁজার আগে এখানেই ধরা ভালো।

নিচের cell আসল cloud call। ADC লাগবে। `DefaultCredentialsError` এলে আগের notebook-এর তিন ধাপের একটাও মেলেনি।

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

root = Path.cwd().resolve()
for candidate in (root, *root.parents):
    if (candidate / "myhr_rag" / "document_loader.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from google.cloud import storage

from myhr_rag import config
from myhr_rag.document_loader import load_processed_documents_from_gcs

client = storage.Client(project=config.PROJECT_ID)
bucket = client.bucket(config.GCS_BUCKET_NAME)

matches = (
    blob for blob in bucket.list_blobs(prefix=config.PROCESSED_HR_PREFIX)
    if blob.name.endswith(".json")
)
blob = next(matches, None)
if blob is None:
    print("কোনো .json নেই। prefix:", config.PROCESSED_HR_PREFIX)
    print("processor.py আগে চালাও, অথবা prefix চেক করো।")
else:
    record = json.loads(blob.download_as_text())
    print(blob.name)
    print(list(record.keys()))
    print(record["text"][:200])

docs = load_processed_documents_from_gcs()
print(f"loaded {len(docs)} documents")
print(Counter(d.metadata["policy_category"] for d in docs))